# Integration & Expectation

## What's covered

- **Integration as area** under a curve, with Riemann sums for intuition
- The **fundamental theorem of calculus** — integration and differentiation as inverse operations
- **Substitution** (change of variables) — the integration analogue of the chain rule
- **Multivariable integration** — briefly, since multivariate change of variables matters for ML
- **Expectation as integral** — the bridge into probability
- **Change of variables for probability density** — the math underlying **normalizing flows**
- **Monte Carlo integration** — when you can't do the integral, sample
- Where this appears in ML — KL divergence, ELBO, normalizing flows, Bayesian methods


## Integration as area

If derivatives measure how a function *changes*, integrals measure how a function *accumulates*. The defining picture: the **definite integral** of `f` from `a` to `b`,

$$
\int_a^b f(x) \, dx
$$

is the **signed area** between the curve `f` and the x-axis on the interval `[a, b]`. "Signed" because area below the axis counts negatively. Formally, we approximate this area by chopping `[a, b]` into thin rectangles and summing — a **Riemann sum**:

$$
\int_a^b f(x) \, dx = \lim_{n \to \infty} \sum_{i=1}^n f(x_i^*) \, \Delta x
$$

where `Δx = (b - a) / n` is the rectangle width and `x_i^*` is any point in the `i`-th sub-interval. As `n → ∞` the sum converges to the exact area.

Just as limits gave us the derivative, limits of sums give us the integral. They are the two sides of calculus.

**For ML:** every time you write down an expectation `E[g(X)] = ∫ g(x) p(x) dx`, you are accumulating values of `g` weighted by the density `p`. Integrals are how we average over continuous distributions.


In [ ]:
import numpy as np

# Numerical integration: estimate the area under f(x) = x^2 from 0 to 2 (true value: 8/3 ≈ 2.6667)
def f(x): return x ** 2

true_value = 8 / 3
print(f"True integral of x^2 from 0 to 2: {true_value:.6f}\n")

# Riemann sum with midpoints — converges as n grows
print(f"{'n':<8} {'Riemann sum':>14} {'error':>10}")
for n in [10, 100, 1000, 10000]:
    xs = np.linspace(0, 2, n + 1)
    mids = 0.5 * (xs[:-1] + xs[1:])
    dx = 2 / n
    riemann = np.sum(f(mids) * dx)
    print(f"{n:<8} {riemann:>14.6f} {abs(riemann - true_value):>10.2e}")

# NumPy's trapezoidal rule does the same accumulation more cleanly
xs = np.linspace(0, 2, 1001)
trapz = np.trapezoid(f(xs), xs)
print(f"\nnp.trapezoid (n=1000):  {trapz:.6f}")


## The fundamental theorem of calculus

The two halves of calculus — differentiation and integration — are bound together by the **fundamental theorem of calculus** (FTC).

**Part 1 — Integration undoes differentiation.** Define `F(x) = ∫_a^x f(t) dt`. Then

$$
F'(x) = f(x)
$$

The derivative of "the area accumulated so far" is just the height of the curve at the right edge. Picture the area function growing as you slide the right edge to the right.

**Part 2 — Differentiation undoes integration.** If `F` is *any* antiderivative of `f` (i.e., `F'(x) = f(x)`), then

$$
\int_a^b f(x) \, dx = F(b) - F(a)
$$

So to compute an integral, find a function whose derivative is `f`, evaluate it at the endpoints, subtract. This is how every elementary integral you've ever computed was done.

**Practical implication.** Integration and differentiation are inverse operations in the same sense that addition and subtraction are. The catch: many functions have derivatives you can write down in closed form but **no closed-form antiderivative**. `e^{-x^2}`, the Gaussian, is the classic example — it has no elementary antiderivative, but its integral is the foundation of probability theory. Numerical integration (or special functions like the error function) handles this.

For ML: most loss functions are easy to differentiate but hard to integrate. That's why training is gradient-based (differentiation), while inference and Bayesian methods rely on tricks (sampling, variational approximations) to avoid intractable integrals.


## Substitution — the chain rule, in reverse

The **substitution rule** is what you get by reading the chain rule backwards. If `u = g(x)` and `du = g'(x) dx`, then

$$
\int f(g(x)) g'(x) \, dx = \int f(u) \, du
$$

In words: spot the form `f(g(x)) \cdot g'(x)` inside an integrand, substitute `u = g(x)`, and the integral becomes simpler.

**Example.** Integrate `2x e^{x^2}`. Let `u = x^2`, so `du = 2x \, dx`. Then

$$
\int 2x e^{x^2} \, dx = \int e^u \, du = e^u + C = e^{x^2} + C
$$

The differentials' bookkeeping (`du = g'(x) dx`) is the same "cancellation" that makes Leibniz notation pop in the chain rule.

**Change of variables (the multivariable upgrade).** Generalize to integrals over regions in `R^n`. If `\Phi: R^n \to R^n` is a smooth bijection (a *coordinate change*), then for any integrable `f`:

$$
\int_{\Phi(\Omega)} f(\mathbf{y}) \, d\mathbf{y} = \int_{\Omega} f(\Phi(\mathbf{x})) \, |\det J_\Phi(\mathbf{x})| \, d\mathbf{x}
$$

The `|det J_Φ|` factor compensates for how the coordinate change locally stretches volumes. We saw exactly this in the linear algebra repo: `|det A|` is the volume scaling factor of a linear map.

This formula is the engine of **normalizing flows** in deep learning — the next-to-last bullet in this notebook.


In [ ]:
# Verify substitution numerically: ∫ 2x e^(x^2) dx from 0 to 1 = e - 1
def integrand(x): return 2 * x * np.exp(x ** 2)

xs = np.linspace(0, 1, 1001)
numerical = np.trapezoid(integrand(xs), xs)
analytic  = np.exp(1) - 1

print(f"numerical integral: {numerical:.6f}")
print(f"analytic   (e - 1): {analytic:.6f}")
print(f"abs error         : {abs(numerical - analytic):.2e}")


## Expectation — the integration most-used in ML

The **expectation** (or *expected value*, or *mean*) of a random variable `X` with probability density `p(x)` is:

$$
\mathbb{E}[X] = \int x \, p(x) \, dx
$$

For a function `g` of that random variable:

$$
\mathbb{E}[g(X)] = \int g(x) \, p(x) \, dx
$$

This is *the* operation of probability and statistics. Variance, covariance, KL divergence, cross-entropy, evidence — all defined as expectations.

**A few important expectations to recognize.**

- **Mean of a Gaussian.** If `X ~ N(μ, σ²)`, then `E[X] = μ`. The Gaussian density integrated against `x` returns the mean exactly. (Computed analytically by completing the square.)
- **Variance.** `Var(X) = E[(X - μ)^2] = E[X^2] - μ^2`. Both `E[X^2]` and `E[X]` are themselves integrals against `p(x)`.
- **Expectation of a constant.** `E[c] = c`. (Trivial — the integral of `c · p(x)` is `c · ∫ p = c`.)
- **Linearity.** `E[aX + bY] = a E[X] + b E[Y]` always — even when `X` and `Y` are dependent. Hard to overstate how often this matters.
- **Tower property.** `E[E[X | Y]] = E[X]`. Conditional expectation iterated gives back the unconditional one.

For ML, every loss you optimize is "approximately" an expectation: `L(w) = E_{(x, y) ~ data}[ℓ(w; x, y)]`. The training set is a Monte Carlo sample from this expectation (next section).


In [ ]:
# Numerical expectation: E[X] and E[X^2] for a standard normal N(0, 1)
# True values: E[X] = 0, E[X^2] = 1, Var(X) = 1.

def gaussian(x, mu=0, sigma=1):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

xs = np.linspace(-8, 8, 5000)
p  = gaussian(xs)

EX  = np.trapezoid(xs * p, xs)
EX2 = np.trapezoid(xs ** 2 * p, xs)
var = EX2 - EX ** 2
print(f"E[X]   = {EX:.4f}     (analytic 0)")
print(f"E[X^2] = {EX2:.4f}     (analytic 1)")
print(f"Var    = {var:.4f}     (analytic 1)")

# Sanity check the density integrates to 1
print(f"\n∫ p(x) dx = {np.trapezoid(p, xs):.6f}   (should be 1)")


## Change of variables for probability density

A core trick in modern generative modeling. If `X` has density `p_X(x)` and `Y = f(X)` is a smooth, invertible function, then `Y` has density:

$$
p_Y(\mathbf{y}) = p_X(f^{-1}(\mathbf{y})) \, \left| \det J_{f^{-1}}(\mathbf{y}) \right|
$$

Equivalently, if we write `x = f^{-1}(y)`:

$$
p_Y(\mathbf{y}) = p_X(\mathbf{x}) \, \left| \det J_f(\mathbf{x}) \right|^{-1}
$$

In log form (which is what's used in practice):

$$
\log p_Y(\mathbf{y}) = \log p_X(\mathbf{x}) - \log \left| \det J_f(\mathbf{x}) \right|
$$

**Why this is huge.** A **normalizing flow** is a stack of smooth invertible neural-network layers `f = f_n \circ \dots \circ f_1` that maps a simple base distribution (typically standard Gaussian) to a complex data distribution. The likelihood of any data point under the model is computed *exactly* using this formula — no sampling, no approximation. The catch is the `log |det J|` term, which must be tractable; architectures like RealNVP, Glow, and neural spline flows are engineered specifically so this determinant is cheap (triangular Jacobian, etc.).

This is the cleanest place in ML where multivariable calculus shows up *exactly* — no approximations.


## Monte Carlo integration

When you can't compute an integral analytically and you can't even do it well numerically (e.g., high-dimensional integrals, where grid-based methods explode), you can **sample**.

If you have i.i.d. samples `X_1, X_2, \dots, X_n \sim p`, then by the law of large numbers:

$$
\frac{1}{n} \sum_{i=1}^n g(X_i) \xrightarrow{n \to \infty} \mathbb{E}_{X \sim p}[g(X)] = \int g(x) p(x) \, dx
$$

This is **Monte Carlo integration**. The error shrinks like `O(1/√n)` — slow but *independent of dimension*. That's why MC integration is the default for high-dimensional problems.

**The biggest ML examples.**

- **Stochastic gradient descent itself.** A mini-batch is a Monte Carlo sample of the loss expectation `E_{(x, y) ~ data}[ℓ]`. The gradient computed from one batch is an unbiased estimator of the true gradient.
- **REINFORCE / policy gradients.** `∇_θ E_{a ~ π_θ}[R(a)]` is estimated by sampling actions and weighting by the log-policy gradient.
- **Variational inference and the reparameterization trick.** `E_{z ~ q_φ(z|x)}[\log p(x | z)]` is estimated by sampling `z` — and the *reparameterization trick* lets you back-propagate through the sampling step.
- **Diffusion models.** Training loss is an expectation over noise scales and over noise samples — a double Monte Carlo.

**Variance reduction** is the practical art of MC integration: importance sampling, control variates, antithetic sampling, common random numbers — all techniques to lower the variance of the estimator and require fewer samples for the same accuracy.


In [ ]:
# Estimate E[X^2] for standard normal by Monte Carlo — true value is 1
rng = np.random.default_rng(0)

print(f"{'n':<10} {'MC estimate':>14} {'error':>10}")
for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
    samples = rng.standard_normal(size=n)
    est = (samples ** 2).mean()
    print(f"{n:<10} {est:>14.6f} {abs(est - 1):>10.2e}")

# Note the error drops by ~3.16x (= sqrt(10)) each time n grows 10x.
print("\nNotice the error shrinks like 1/sqrt(n), the Monte Carlo rate.")


## Where this appears in ML

Integration is the bridge between calculus and probability — and probability is everywhere in ML.

- **Expectation losses.** `L(w) = E[ℓ(w; x, y)]` for `(x, y)` from the data distribution. Stochastic gradient descent samples this expectation; each mini-batch is a Monte Carlo estimate.
- **KL divergence.** `KL(p || q) = \int p(x) \log (p(x) / q(x)) dx`. Used in variational inference, distillation, language modeling (cross-entropy = KL + entropy), reinforcement learning (TRPO, PPO have KL constraints).
- **Cross-entropy and log-likelihood.** `\log p(x | \theta)` is the integrand in maximum-likelihood training of any probabilistic model.
- **Evidence Lower Bound (ELBO).** Used in VAEs and variational inference: `ELBO(φ) = E_{q_φ}[\log p(x, z) - \log q_φ(z)]`. The expectation is estimated by sampling.
- **Normalizing flows.** Exact likelihood via the change-of-variables formula. Glow, RealNVP, neural spline flows, FFJORD all rest on this.
- **Diffusion models.** Forward process: integrate stochastic differential equations adding noise. Reverse process: integrate (or sample from) the score-based reverse SDE. Training loss is an expectation, computed via MC.
- **Bayesian inference.** Marginal likelihood `p(D) = \int p(D|θ) p(θ) dθ` is an integral over parameters. Almost always intractable; tackled by MCMC, variational inference, or Laplace approximation (notebook 6).
- **Importance sampling.** Estimate `E_p[f(X)]` by sampling from a different distribution `q` and reweighting by `p/q`. Foundational for off-policy reinforcement learning.
- **Score-based generative modeling.** The score `\nabla_x \log p(x)` is the gradient of a log-density; sampling from `p` is integrating a stochastic process driven by the score.
- **Probabilistic programming (Pyro, NumPyro, Stan).** Express models, compile to gradients of log-likelihoods, integrate by MCMC or variational inference.

That closes the calculus repo. From `lim h→0 (f(x+h) - f(x))/h` in notebook 1 to normalizing flows in notebook 8.

Next on the math foundations roadmap: **probability and statistics** — distributions, MLE, expectation made formal, hypothesis testing, and the probabilistic vocabulary that all of modern ML is written in. Then the **machine learning** repo finally builds on top of all three pillars: linear algebra, calculus, and probability.
